In [ ]:
# pip install python-dotenv openAI langchain langchain_openai langchain-community

In [1]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [2]:
# <호출(invoke) 출력>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../env")
# 오픈AI의 대규모 언어 모델 설정
model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 주어진 주제에 대한 설명 요청
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧게 설명을 해주세요")
# 출력 파서 정의: AI 메시지의 출력 내용을 추출
parser = StrOutputParser()
# 프롬프트, 모델, 출력 파서를 체인으로 연결
chain = prompt | model | parser

# 응답을 호출
chain.invoke({"topic":"더블딥"})


'더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기침체가 두 차례에 걸쳐 발생하는 현상을 의미합니다. 첫 번째 경기침체 후 경제가 일시적으로 회복되다가 다시 침체에 빠지는 경우를 나타냅니다. 이러한 상황은 소비자 신뢰가 회복되지 않거나, 고용 시장이 회복되지 않는 등 여러 요인으로 인해 발생할 수 있습니다. 더블딥은 경제 전반에 부정적인 영향을 미치며, 정책 입안자에게도 큰 도전 과제가 됩니다.'

In [3]:
# <배치(Batch) 출력>

# 주어진 주제 리스트를 배치로 출력
chain.batch([{"topic": "더블딥"}, {"topic": "Nvidia Isaac Sim"}])

['더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 후퇴가 일어난 후 잠시 회복세를 보였다가 다시 경기 침체에 빠지는 상황을 의미합니다. 즉, 경제가 두 번의 침체를 겪는 형태로, 첫 번째 침체에서 반등한 것처럼 보이다가 다시 악화되는 현상을 설명합니다. 더블딥은 주로 생산, 고용, 소비 등의 지표에서 나타나며, 경제 정책에 대한 논의에서 중요한 개념으로 다뤄집니다.',
 'Nvidia Isaac Sim은 로봇 시뮬레이션을 위한 고급 플랫폼으로, Nvidia의 Omniverse 기술을 기반으로 합니다. 이 플랫폼은 로봇 개발자와 연구자들이 현실적인 물리 및 환경에서 로봇을 훈련하고 테스트할 수 있도록 돕습니다. Isaac Sim은 3D 그래픽, 인공지능(AI), 심리스 가상 현실(VR) 및 고성능 컴퓨팅을 통합하여 실제처럼 시뮬레이션 환경을 제공합니다. 이를 통해 로봇 알고리즘의 개발, 테스트, 검증 과정이 더 효율적이고 안전하게 이루어질 수 있습니다. Isaac Sim은 드론, 자율주행차, 산업용 로봇 등 다양한 분야에서 활용됩니다.']

In [4]:
# <스트림(Stream) 출력>

# 응답을 토큰 단위로 스트리밍하여 출력
for token in chain.stream({"topic":"Nvidia Isaac Sim"}):
    # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며, 버퍼를 즉시 플러시하여 실시간으로 보여줌
    print(token, end="", flush=True)


Nvidia Isaac Sim은 로봇 시뮬레이션 및 AI 개발을 위한 플랫폼으로, Nvidia의 Omniverse 기술을 기반으로 합니다. 이 플랫폼은 현실적인 3D 환경에서 로봇을 설계하고 테스트할 수 있도록 돕습니다. 또한, 강력한 물리 엔진과 고급 그래픽을 사용하여 실제와 유사한 시뮬레이션을 제공하며, 로봇의 센서 및 행동을 시뮬레이션할 수 있는 다양한 도구와 기능을 제공합니다. 이로 인해 연구자와 개발자들은 다양한 상황에서 로봇을 교육하고 평가할 수 있어, 실제 환경에서의 안전성을 검증하는 데 큰 도움이 됩니다.

In [5]:
# <구성된 체인을 다른 러너블과 결합하기>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# "이 대답을 영어로 번역해 주세요"라는 질문을 생성하는 프롬프트 템플릿을 정의
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 이전에 정의된 체인(chain)을 사용하여 대답을 생성하고, 그 대답을 영어로 번역하도록 프롬프트에 전달한 후, 모델을 통해 결과를 생성하여 최종적으로 문자열로 파싱하는 체인을 구성
composed_chain = {"answer": chain} | analysis_prompt | model | StrOutputParser()
# "더블딥"이라는 주제로 대답을 생성하고, 체인 실행
composed_chain.invoke({"topic": "Nvidia Isaac Sim"})

"Nvidia Isaac Sim is an advanced platform for robot simulation and development. This software helps design and test robots in realistic 3D environments. It leverages Nvidia's powerful GPUs to support realistic physics simulation, sensor modeling, path planning, and testing of artificial intelligence algorithms. Additionally, Isaac Sim is integrated with ROS (Robot Operating System), making it easier for robot developers to interface with real robots. This contributes to accelerating research and development in robotics and broadening the potential applications across various industries."

In [6]:
# <람다 함수를 사용한 체인을 통해 구성하기>

# 이전에 정의된 값들
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
chain = prompt | model | StrOutputParser()
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 람다 함수를 사용한 체인 구성
composed_chain_with_lambda = (
    # 이전에 정의된 체인(chain)을 사용하여 입력된 데이터를 받아옵니다.
    chain
    # 입력된 데이터를 "answer" 키로 변환하는 람다 함수를 적용합니다.
    | (lambda input: {"answer": input})
    # "answer " 키를 가진 데이터를 영어로 번역하도록 프롬프트에 전달합니다.
    | analysis_prompt
    # 프롬프트에서 생성된 요청을 모델에 전달하여 결과를 생성합니다.
    | model
    # 모델에서 반환된 결과를 문자열로 파싱합니다.
    | StrOutputParser()
)
# "더블딥"라는 주제로 대답을 생성하고, 그 대답을 영어로 번역합니다.
composed_chain_with_lambda.invoke({"topic": "Nvidia Isaac Sim"})


'Nvidia Isaac Sim is a platform for robot simulation and artificial intelligence (AI) development. This software provides a high-performance physics engine and realistic graphics, allowing developers to create a virtual environment to test and optimize the movements, recognition, and interactions of robots. Isaac Sim offers tools for simulating various robotic scenarios and training AI models, accelerating the development of real-world robotic applications. Additionally, it is compatible with ROS (Robot Operating System), making it easily accessible for researchers and developers.'

In [7]:
# <`.pipe()`를 통해 체인 구성하기>

# (방법1) 여러 작업을 순차적으로 .pipe를 통해 연결하여 체인 구성하기
composed_chain_with_pipe = (
  # 이전에 정의된 체인(chain)으로 입력된 데이터를 받아옴
  chain
  # 입력된 데이터를 "answer" 키로 변환하는 람다 함수 적용
  .pipe(lambda input: {"answer": input})
  # analysis_prompt를 체인에 연결하여 설명을 영어로 번역하는 작업 추가
  .pipe(analysis_prompt)
  # 모델을 사용해 응답 생성
  .pipe(model)
  # 생성된 응답을 문자열로 파싱
  .pipe(StrOutputParser())
)
# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})


'"Double dip" is a term primarily used in economics or financial markets to refer to a situation where a specific asset or the economy experiences two consecutive periods of recession or decline. For example, it refers to a scenario where the economy fails to recover and falls back into recession. Generally, a double dip indicates uncertainty in economic recovery and can raise significant concerns among investors or policymakers.'

In [8]:
# (방법2) 좀 더 간단하게 연결하기
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())

# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})

'"Double Dip" refers to a phenomenon in the economy where a recovery appears to occur temporarily, only to plunge back into recession. In other words, it describes a situation where the economy seems to be recovering, but then declines again. This phenomenon is primarily caused by the volatility of economic cycles and occurs when temporary improvements in economic indicators or consumer confidence do not sustain and subsequently worsen. Double Dip is a concern for many economists and policymakers.'

In [9]:
# <`RunnableParallel`을 이용한 체인 구성>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧은 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "미국중간선거"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 미국 중간선거는 대통령 선거 중간에 실시되는 선거로, 대개 2년에 한 번 열립니다. 이 선거에서는 미국 하원 전체(435석)와 상원 일부(100석 중 33석 또는 34석)가 선출됩니다. 또한 주지사 선거와 주 및 지방 정부의 다양한 직위가 함께 치러집니다. 중간선거는 대통령의 정치적 입지를 평가할 수 있는 중요한 기회로 여겨지며, 각 당의 의석 수가 바뀔 경우 정치적 균형에 큰 영향을 미칠 수 있습니다.
영어 설명: The U.S. midterm elections are held every four years, midway through a President's term. During these elections, voters elect members of Congress, including all 435 members of the House of Representatives and one-third of the 100 Senate seats. Midterms often serve as a referendum on the sitting president's performance and can result in significant shifts in the political landscape, impacting party control and legislative priorities. They also include gubernatorial elections and various state and local offices.


In [ ]:
# 참고
class CustomLCEL:
    def __init__(self, value):
        self.value = value  # 객체 생성 시 값을 초기화합니다.
    def __or__(self, other):
        if callable(other):
            return CustomLCEL(other(self.value))  # other가 함수일 경우, 함수를 호출하고 그 결과를 새로운 객체로 반환합니다.
        else:
            raise ValueError("Right operand must be callable")  # other가 함수가 아니면 에러를 발생시킵니다.
    def result(self):
        return self.value  # 현재 값을 반환합니다.
# 문자열 끝에 느낌표를 추가하는 함수
def add_exclamation(s):
    return s + "!"
# 문자열을 뒤집는 함수
def reverse_string(s):
    return s[::-1]
# 파이프라인을 생성하여 순차적으로 문자열 변환 작업을 수행합니다.
custom_chain = (
    CustomLCEL("랭체인 공부하기")  # "랭체인 공부하기"로 초기화된 객체 생성
    | add_exclamation  # 느낌표 추가
    | reverse_string  # 문자열 뒤집기
)
# 최종 결과를 출력합니다.
result = custom_chain.result()
print(result)
# 출력: !기하부공 인체랭


!기하부공 인체랭
